# MTG Deck Card Predictor — Optuna Hyperparameter Sweep

Automated search for optimal hyperparameters using Bayesian optimization.
Optuna tries many configurations, prunes unpromising trials early, and reports
which parameters matter most.

**Architecture**: HeteroGNN + MLP-scored Autoregressive Deck Constructor with argmax selection,
count teacher forcing, and separate count classification heads for non-basic (1–4) and basic lands (1–20).
Multi-format support (Standard + Pioneer) with detached GNN backward and gradient checkpointing.

**Primary metric**: Jaccard similarity between predicted and actual decklist.

**Runtime**: Select `Runtime > Change runtime type > A100 GPU` before running.

**Important**: After mounting Drive and installing dependencies, **restart the runtime** before
running the sweep to ensure the latest code is loaded.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow optuna

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')

In [ ]:
# Stage data files from Google Drive to Colab's local disk for faster I/O.
# Critical for sweeps: graph.pt is loaded once per trial, and Drive reads
# add ~10-15s each time. Over 200 trials that's ~30-50 minutes saved.
#
# IMPORTANT: If you've updated source files on Drive, restart the runtime
# (Runtime > Restart runtime) before re-running from this cell.

import shutil
from pathlib import Path
from src.config import GRAPH_PATH, DECKLISTS_PARQUET, METAGAME_PARQUET, CARDS_PARQUET, ACTIVE_FORMATS, format_path

LOCAL_DATA = Path('/content/mtg_data')

# Set FORCE_RESTAGE = True if you've rebuilt graph.pt or other data files
# since the last time you ran this notebook. This deletes the local cache
# and re-copies everything from Drive.
FORCE_RESTAGE = True

if FORCE_RESTAGE and LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
    print('Cleared stale local cache.')

LOCAL_DATA.mkdir(parents=True, exist_ok=True)

# Stage shared files
for src_file in [GRAPH_PATH, CARDS_PARQUET]:
    dst = LOCAL_DATA / src_file.name
    if not dst.exists():
        shutil.copy2(src_file, dst)
        print(f'  Copied {src_file.name} ({src_file.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'  Already staged: {src_file.name}')

# Stage per-format files (decklists + metagame)
for fmt in ACTIVE_FORMATS:
    for parquet_base in [DECKLISTS_PARQUET, METAGAME_PARQUET]:
        fmt_src = format_path(parquet_base, fmt)
        if fmt_src.exists():
            fmt_dst = LOCAL_DATA / fmt / fmt_src.name
            fmt_dst.parent.mkdir(parents=True, exist_ok=True)
            if not fmt_dst.exists():
                shutil.copy2(fmt_src, fmt_dst)
                print(f'  Copied {fmt}/{fmt_src.name} ({fmt_src.stat().st_size / 1e6:.1f} MB)')
            else:
                print(f'  Already staged: {fmt}/{fmt_src.name}')

# Patch config paths so train_deck uses local copies.
import src.config as cfg
cfg.GRAPH_PATH = LOCAL_DATA / 'graph.pt'
cfg.CARDS_PARQUET = LOCAL_DATA / 'cards.parquet'
cfg.DECKLISTS_PARQUET = LOCAL_DATA / 'decklists.parquet'
cfg.METAGAME_PARQUET = LOCAL_DATA / 'metagame.parquet'

# Print graph size to verify it's the right file
graph_size = (LOCAL_DATA / 'graph.pt').stat().st_size / 1e6
print(f'\nGRADIENT_CHECKPOINTING: {cfg.GRADIENT_CHECKPOINTING}')
print(f'Graph size: {graph_size:.1f} MB')
print(f'Data staged to {LOCAL_DATA} (local SSD)')
print(f'Active formats: {ACTIVE_FORMATS}')

In [ ]:
# ── Training Mode ──
# 'pretrain': Train on all formats (Standard + Pioneer)
# 'finetune': Fine-tune a pretrained model on a single format
TRAINING_MODE = 'pretrain'          # 'pretrain' or 'finetune'
FORMAT_FILTER = 'standard'          # Target format for finetune mode
                                    # Options: 'standard', 'pioneer'
PRETRAINED_PATH = None              # Path to pretrained checkpoint (finetune only)

# ── Search Space Configuration (edit these) ──
# NOTE: With the expanded Pioneer+Standard graph (~31M edges), larger configs
# use significant GPU memory. The diagnostic cell above shows peak memory at
# d_model=64. Scale accordingly:
#   d_model=64,  1 layer  → ~30 GB peak
#   d_model=128, 2 layers → ~60 GB peak
#   d_model=256, 2 layers → ~80+ GB peak (likely OOM on A100)
# Keep d_model * num_gnn_layers product reasonable for your GPU.

N_TRIALS = 200
STUDY_NAME = 'deck-sweep'

SEARCH_SPACE = {
    'd_model':           [64, 128],
    'd_message':         [64, 128],
    'd_count':           [8, 16, 32],
    'num_gnn_layers':    {'low': 1, 'high': 2},
    'dropout':           {'low': 0.05, 'high': 0.30, 'step': 0.05},
    'learning_rate':     {'low': 1e-5, 'high': 1e-2},
    'weight_decay':      {'low': 1e-6, 'high': 1e-3},
    'count_loss_weight': {'low': 0.1, 'high': 3.0, 'step': 0.1},
    'warmup_epochs':     [0, 3, 5, 10],
    'lr_scheduler':      ['cosine', 'linear', 'none'],
    'batch_size':        [0, 4, 1],           # archetypes per step (0 = all)
    'batch_strategy':    ['random', 'color_balanced'],
    'color_loss_weighting': [True, False],
    'gnn_lr_scale':      [0.01, 0.1, 0.3, 0.5, 1.0],
    # Curriculum negative sampling
    'curriculum_neg_start':  {'low': 50, 'high': 500, 'step': 50},
    'curriculum_neg_epochs': {'low': 5, 'high': 40, 'step': 5},
    'curriculum_neg_schedule': ['linear', 'exponential'],
}

FIXED_PARAMS = {
    'patience': 10,
    'num_epochs': 100,
    'recency_days': 30,
    'top_n_archetypes': 0,    # 0 = train on all archetypes; N = top N per format
    'n_val_archetypes': 3,
    'n_cv_folds': 5,
    'val_every': 10,
    'edge_chunk_size': 2_000_000,  # memory optimization (0 = no chunking)
    'mode': TRAINING_MODE,
}

if TRAINING_MODE == 'finetune':
    FIXED_PARAMS['format_filter'] = FORMAT_FILTER
    FIXED_PARAMS['pretrained_path'] = PRETRAINED_PATH
    from src.config import TRANSFER_FINETUNE_EPOCHS
    FIXED_PARAMS['num_epochs'] = TRANSFER_FINETUNE_EPOCHS

N_STARTUP_TRIALS = 10
N_WARMUP_STEPS = 3

print(f'Training mode: {TRAINING_MODE}')
print(f'Search: {N_TRIALS} trials over {len(SEARCH_SPACE)} parameters')
print(f'Pruning: starts after {N_STARTUP_TRIALS} trials, warmup {N_WARMUP_STEPS} steps')
print(f'Fixed: {FIXED_PARAMS}')

In [ ]:
# ── Pre-sweep memory diagnostic ──
# Run a single mini-trial to verify training works before starting the full sweep.
# This catches OOM early and shows GPU memory at each stage.

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from pathlib import Path
from src.train_deck import train_deck

LOCAL_RESULTS = Path('/content/sweep_tmp')
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

print('Running memory diagnostic (d_model=64, 1 epoch, 1 fold)...\n')
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f'GPU before: {torch.cuda.memory_allocated()/1e9:.1f} GB')

try:
    diag_result = train_deck(
        device=device,
        d_model=64, d_message=64, d_count=8,
        num_gnn_layers=1, dropout=0.1,
        learning_rate=1e-4, gnn_lr_scale=0.1,
        weight_decay=1e-5, count_loss_weight=1.0,
        warmup_epochs=0, lr_scheduler='none',
        batch_size=1, n_cv_folds=1, n_val_archetypes=3,
        num_epochs=1, patience=10, recency_days=30,
        val_every=1, mode=TRAINING_MODE,
        edge_chunk_size=FIXED_PARAMS.get('edge_chunk_size', 2_000_000),
        results_base=LOCAL_RESULTS,
    )
    if torch.cuda.is_available():
        print(f'\nGPU after:  {torch.cuda.memory_allocated()/1e9:.1f} GB')
        print(f'GPU peak:   {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
    print('Diagnostic PASSED -- sweep should work.\n')
    del diag_result
    import gc; gc.collect()
    torch.cuda.empty_cache()
except torch.cuda.OutOfMemoryError:
    if torch.cuda.is_available():
        print(f'\nGPU peak:   {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
    print('Diagnostic FAILED -- OOM even with smallest config!')
    print('The graph may be too large for this GPU. Check graph.pt size and edge counts.')

In [ ]:
import optuna
import time
import logging
import shutil
import gc
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('src.train_deck').setLevel(logging.WARNING)

# Verify config paths are patched (from staging cell above)
import src.config as cfg
print(f'GRADIENT_CHECKPOINTING: {cfg.GRADIENT_CHECKPOINTING}')
print(f'GRAPH_PATH: {cfg.GRAPH_PATH}')
print(f'DECKLISTS_PARQUET: {cfg.DECKLISTS_PARQUET}')

from src.train_deck import train_deck

# Use Colab's local disk for intermediate trial results (fast, no Drive I/O)
LOCAL_RESULTS = Path('/content/sweep_tmp')
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

trial_results = {}


def objective(trial):
    """Optuna objective: sample hyperparameters, train, return best val Jaccard."""
    params = dict(FIXED_PARAMS)

    params['d_model'] = trial.suggest_categorical('d_model', SEARCH_SPACE['d_model'])
    params['d_message'] = trial.suggest_categorical('d_message', SEARCH_SPACE['d_message'])
    params['d_count'] = trial.suggest_categorical('d_count', SEARCH_SPACE['d_count'])
    params['num_gnn_layers'] = trial.suggest_int('num_gnn_layers', **SEARCH_SPACE['num_gnn_layers'])
    params['dropout'] = trial.suggest_float('dropout', **SEARCH_SPACE['dropout'])
    params['learning_rate'] = trial.suggest_float('learning_rate', **SEARCH_SPACE['learning_rate'], log=True)
    params['weight_decay'] = trial.suggest_float('weight_decay', **SEARCH_SPACE['weight_decay'], log=True)
    params['count_loss_weight'] = trial.suggest_float('count_loss_weight', **SEARCH_SPACE['count_loss_weight'])
    params['warmup_epochs'] = trial.suggest_categorical('warmup_epochs', SEARCH_SPACE['warmup_epochs'])
    params['lr_scheduler'] = trial.suggest_categorical('lr_scheduler', SEARCH_SPACE['lr_scheduler'])
    params['batch_size'] = trial.suggest_categorical('batch_size', SEARCH_SPACE['batch_size'])
    params['batch_strategy'] = trial.suggest_categorical('batch_strategy', SEARCH_SPACE['batch_strategy'])
    params['color_loss_weighting'] = trial.suggest_categorical('color_loss_weighting', SEARCH_SPACE['color_loss_weighting'])
    params['gnn_lr_scale'] = trial.suggest_categorical('gnn_lr_scale', SEARCH_SPACE['gnn_lr_scale'])

    # Curriculum negative sampling
    params['curriculum_neg_start'] = trial.suggest_int('curriculum_neg_start', **SEARCH_SPACE['curriculum_neg_start'])
    params['curriculum_neg_epochs'] = trial.suggest_int('curriculum_neg_epochs', **SEARCH_SPACE['curriculum_neg_epochs'])
    params['curriculum_neg_schedule'] = trial.suggest_categorical('curriculum_neg_schedule', SEARCH_SPACE['curriculum_neg_schedule'])

    # Write intermediate results to local disk, not Google Drive
    params['results_base'] = LOCAL_RESULTS

    start = time.time()
    try:
        result = train_deck(device=device, trial=trial, **params)
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        torch._dynamo.reset()
        print(f'  Trial {trial.number}: OOM. Skipping.')
        raise optuna.TrialPruned()

    elapsed = time.time() - start

    fold_0 = result['fold_results'][0]
    best_metric = fold_0['best_val_metric']

    val_hist = fold_0.get('val_metrics_history', [])
    last_val = val_hist[-1] if val_hist else {}
    precision = last_val.get('precision', 0)

    # Save only lightweight metadata — NOT the model or graph data.
    # The full result['data'] holds the HeteroData graph on GPU, and
    # result['model'] holds the trained model. Keeping these across
    # trials leaks GPU memory and causes OOM on subsequent trials.
    trial_results[trial.number] = {
        'best_val_metric': best_metric,
        'best_epoch': fold_0['best_epoch'],
        'precision': precision,
        'elapsed_s': elapsed,
        'hp': result['hp'],
        'fold_results': [{
            'fold': fold_0['fold'],
            'train_archetypes': fold_0['train_archetypes'],
            'val_archetypes': fold_0['val_archetypes'],
            'best_epoch': fold_0['best_epoch'],
            'best_val_metric': fold_0['best_val_metric'],
            'train_losses': fold_0['train_losses'],
            'val_metrics_history': fold_0['val_metrics_history'],
            'model_path': fold_0['model_path'],
        }],
        'run_dir': result['run_dir'],
    }

    bs_label = 'all' if params['batch_size'] == 0 else str(params['batch_size'])
    clw_label = 'T' if params['color_loss_weighting'] else 'F'
    print(f'  Trial {trial.number}: Jacc={best_metric:.4f} Prec={precision:.3f} '
          f'(epoch {fold_0["best_epoch"]}, {elapsed:.0f}s) '
          f'| dm={params["d_model"]} dg={params["d_message"]} dc={params["d_count"]} '
          f'ly={params["num_gnn_layers"]} bs={bs_label} '
          f'strat={params["batch_strategy"]} clw={clw_label} '
          f'lr={params["learning_rate"]:.1e} glr={params["gnn_lr_scale"]} '
          f'do={params["dropout"]:.2f} closs={params["count_loss_weight"]:.1f} '
          f'cns={params["curriculum_neg_start"]} cne={params["curriculum_neg_epochs"]} '
          f'csch={params["curriculum_neg_schedule"]}')

    # Aggressively free GPU memory before next trial
    del result
    gc.collect()
    torch.cuda.empty_cache()
    torch._dynamo.reset()

    return best_metric


pruner = optuna.pruners.MedianPruner(
    n_startup_trials=N_STARTUP_TRIALS,
    n_warmup_steps=N_WARMUP_STEPS,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=pruner,
)

print(f'\nStarting {N_TRIALS}-trial sweep...')
print(f'Intermediate results: {LOCAL_RESULTS} (local disk)\n')
sweep_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, catch=(Exception,))
sweep_elapsed = time.time() - sweep_start

n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_failed = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

print(f'\n{"=" * 60}')
print(f'Sweep complete: {len(study.trials)} trials in {sweep_elapsed/60:.1f} min')
print(f'  Completed: {n_complete}  Pruned: {n_pruned}  Failed: {n_failed}')
if n_complete > 0:
    print(f'\nBest trial #{study.best_trial.number}:')
    print(f'  Jaccard = {study.best_value:.4f}')
    for k, v in study.best_params.items():
        print(f'  {k}: {v}')
else:
    print('\nNo trials completed. Check error logs above.')

In [ ]:
import matplotlib.pyplot as plt

# Optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Trial values over time
ax = axes[0]
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed]
trial_vals = [t.value for t in completed]
best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]

ax.scatter(trial_nums, trial_vals, alpha=0.5, s=20, label='Trial value')
ax.plot(trial_nums, best_so_far, 'r-', linewidth=2, label='Best so far')

pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
if pruned:
    ax.scatter([t.number for t in pruned], [0] * len(pruned),
              marker='x', color='gray', alpha=0.3, s=15, label='Pruned')

ax.set_xlabel('Trial'); ax.set_ylabel('Jaccard')
ax.set_title('Optimization History'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[1]
try:
    importance = optuna.importance.get_param_importances(study)
    names = list(importance.keys())
    values = list(importance.values())
    colors = plt.cm.viridis([v / max(values) for v in values])
    ax.barh(names, values, color=colors)
    ax.set_xlabel('Importance'); ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except Exception as e:
    ax.text(0.5, 0.5, f'Importance unavailable:\n{e}',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

# Print all completed trials sorted by value
print(f'\nAll completed trials (sorted by Jaccard):')
print(f'{"#":>4}  {"Jacc":>7}  {"Prec":>6}  {"dm":>4}  {"dc":>4}  {"ly":>3}  {"bs":>3}  '
      f'{"lr":>10}  {"glr":>4}  {"do":>5}  {"clw":>5}  {"wu":>3}  {"sched":>7}')
print('-' * 90)
for t in sorted(completed, key=lambda t: t.value, reverse=True):
    p = t.params
    tr = trial_results.get(t.number, {})
    prec = tr.get('precision', 0)
    bs_label = 'all' if p['batch_size'] == 0 else str(p['batch_size'])
    print(f'{t.number:4d}  {t.value:7.4f}  {prec:6.3f}  {p["d_model"]:4d}  {p["d_count"]:4d}  '
          f'{p["num_gnn_layers"]:3d}  {bs_label:>3s}  '
          f'{p["learning_rate"]:10.1e}  {p["gnn_lr_scale"]:4.1f}  {p["dropout"]:5.2f}  '
          f'{p["count_loss_weight"]:5.1f}  {p["warmup_epochs"]:3d}  {p["lr_scheduler"]:>7s}')

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

# ── Configuration ──
TOP_N = 5  # How many top models to save (set to None to save all completed)

# ── Create sweep output directory on Google Drive (final results only) ──
sweep_ts = datetime.now().strftime('%Y-%m-%d_%H%M%S')
sweep_dir = Path(f'results/sweeps/{sweep_ts}')
sweep_dir.mkdir(parents=True, exist_ok=True)

# ── Identify top trials ──
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
ranked = sorted(completed, key=lambda t: t.value, reverse=True)
top_trials = ranked[:TOP_N] if TOP_N else ranked

print(f'Saving top {len(top_trials)} models to Google Drive: {sweep_dir}/\n')

# ── Save each top model (copy from local disk to Drive) ──
for rank, trial in enumerate(top_trials, 1):
    tr = trial_results.get(trial.number)
    if tr is None:
        print(f'  Skipping trial {trial.number} -- results not in memory')
        continue

    fold_0 = tr['fold_results'][0]
    p = trial.params

    # Descriptive folder name
    folder_name = (
        f'rank{rank:02d}_trial{trial.number:02d}'
        f'_jacc-{trial.value:.3f}'
        f'_dm{p["d_model"]}_ly{p["num_gnn_layers"]}'
        f'_lr{p["learning_rate"]:.1e}_do{p["dropout"]:.2f}'
    )
    model_dir = sweep_dir / folder_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # Copy model checkpoint from local disk to Drive
    src_model = Path(fold_0['model_path'])
    if src_model.exists():
        shutil.copy2(src_model, model_dir / 'model_fold0.pt')

    # Copy training log from local disk to Drive
    src_log = tr['run_dir'] / 'training_log.json'
    if src_log.exists():
        shutil.copy2(src_log, model_dir / 'training_log.json')

    # Save config JSON with full hyperparameters + metrics
    config = {
        'rank': rank,
        'trial_number': trial.number,
        'best_epoch': fold_0['best_epoch'],
        'training_time_s': tr['elapsed_s'],
        'hyperparameters': tr['hp'],
        'val_archetypes': fold_0['val_archetypes'],
        'train_archetypes': fold_0['train_archetypes'],
        'best_val_jaccard': trial.value,
        'training_curves': {
            'train_losses': fold_0['train_losses'],
            'val_metrics_history': fold_0['val_metrics_history'],
        },
    }
    with open(model_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2, default=str)

    print(f'  #{rank} Trial {trial.number}: Jaccard={trial.value:.3f} '
          f'(epoch {fold_0["best_epoch"]}, {tr["elapsed_s"]:.0f}s) '
          f'| val={", ".join(fold_0["val_archetypes"])} -> {folder_name}/')

# ── Save sweep summary JSON ──
sweep_summary = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'study_name': STUDY_NAME,
    'n_trials': len(study.trials),
    'n_completed': len(completed),
    'n_pruned': len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
    'elapsed_minutes': sweep_elapsed / 60,
    'top_n_saved': len(top_trials),
    'best_trial': {
        'number': study.best_trial.number,
        'value': study.best_value,
        'params': study.best_params,
    },
    'fixed_params': FIXED_PARAMS,
    'search_space': {k: str(v) for k, v in SEARCH_SPACE.items()},
    'all_trials': [
        {
            'number': t.number,
            'state': t.state.name,
            'value': t.value if t.value is not None else None,
            'params': t.params,
            'duration_s': (t.datetime_complete - t.datetime_start).total_seconds()
                if t.datetime_complete and t.datetime_start else None,
        }
        for t in study.trials
    ],
}

# Add parameter importance
try:
    importance = optuna.importance.get_param_importances(study)
    sweep_summary['param_importance'] = {k: float(v) for k, v in importance.items()}
except Exception:
    pass

with open(sweep_dir / 'sweep_summary.json', 'w') as f:
    json.dump(sweep_summary, f, indent=2)

print(f'\n{"=" * 60}')
print(f'Sweep saved to Google Drive: {sweep_dir}/')
print(f'  sweep_summary.json -- full sweep results ({len(study.trials)} trials)')
print(f'  {len(top_trials)} model directories -- each with model, config')

## Cleanup

Delete the local temporary directory used for intermediate trial results.

In [ ]:
# Clean up local temporary trial directories
# All top models have already been copied to Google Drive in the sweep directory.

import shutil
from pathlib import Path

local_dir = Path('/content/sweep_tmp')
if local_dir.exists():
    n_dirs = len(list(local_dir.rglob('*')))
    shutil.rmtree(local_dir, ignore_errors=True)
    print(f'Cleaned up local temp directory: {local_dir} ({n_dirs} items)')
else:
    print('No local temp directory to clean.')

print(f'\nFinal results on Google Drive: {sweep_dir}/')
!ls "{sweep_dir}"